# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` — ключ `A` встречается два раза, а `B` один раз; `get(c, 0)` берёт старое значение или 0.
2. `[0, 1, 1]` — `append(1)` изменяет тот же список, поэтому после двух вызовов в нём два элемента `1`.
3. `2` — после `lower()` строки будут `almaty`, `almaty`, `astana`, а множество оставит только уникальные значения.
4. `{'ab': 2, 'abc': 3}` — повторный ключ `ab` не создаёт новую запись в словаре.
5. Сначала `5`, затем `None` — `print(n)` выводит число, но функция без `return` возвращает `None`.

In [1]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [2]:
# Загрузка orders.csv: файл должен лежать рядом с ноутбуком.
import os

if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком.")

print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(len(rows), "строк")
print("Поля:", list(rows[0].keys()))

5463 строк
Поля: ['order_id', 'order_date', 'customer_id', 'city', 'category', 'amount', 'delivery_days', 'status']


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [4]:
# База

# 1

def to_num(s):
    s = s.strip()
    s = s.replace(" ", "")
    s = s.replace(",", ".")
    number = float(s)
    if number == int(number):
        return int(number)
    return number


def clean_city(s):
    city = s.strip()
    city = city.lower()
    names = {
        "алматы": "Алматы",       "almaty": "Алматы",
        "астана": "Астана",       "astana": "Астана",
        "караганда": "Караганда", "karaganda": "Караганда",
        "шымкент": "Шымкент",     "shymkent": "Шымкент",
        "актобе": "Актобе",       "aktobe": "Актобе",
        "атырау": "Атырау",       "atyrau": "Атырау",
    }
    return names.get(city, city.capitalize())


print(to_num("1 230"), to_num("8070.0"), to_num(" 19 490 "))
print(clean_city(" Алматы"), clean_city("ALMATY"), clean_city("Astana "), clean_city("шымкент"))

cities_clean = []
for row in rows:
    cities_clean.append(clean_city(row["city"]))
print(set(cities_clean))

1230 8070 19490
Алматы Алматы Астана Шымкент
{'Актобе', 'Караганда', 'Шымкент', 'Атырау', 'Астана', 'Алматы'}


In [5]:
# 2

category_count = {}
for row in rows:
    category = row["category"]
    category_count[category] = category_count.get(category, 0) + 1

def second_item(pair):
    return pair[1]

sorted_categories = sorted(category_count.items(), key=second_item, reverse=True)

for category, count in sorted_categories:
    print(category, count)

Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653


In [6]:
# 3

revenue = {}
paid_count = {}

for row in rows:
    if row["status"] == "оплачен":
        category = row["category"]
        amount = to_num(row["amount"])
        revenue[category] = revenue.get(category, 0) + amount
        paid_count[category] = paid_count.get(category, 0) + 1

print(f"{'Категория':<12}{'Выручка, тг':>14}{'Средний чек, тг':>18}")
for category in sorted(revenue):
    average = revenue[category] / paid_count[category]
    print(f"{category:<12}{revenue[category]:>14,.0f}{average:>18,.2f}")

Категория      Выручка, тг   Средний чек, тг
Дом             10,516,320         10,654.83
Книги            2,294,100          3,934.99
Красота          5,507,620          6,188.34
Одежда          18,321,080         13,285.77
Электроника     65,270,240         61,286.61


In [7]:
# 4

month_count = {}
for row in rows:
    month = row["order_date"][:7]
    month_count[month] = month_count.get(month, 0) + 1

for month in sorted(month_count):
    print(month, month_count[month])

2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [8]:
# Стандарт

# 5

month_revenue = {}
for row in rows:
    if row["status"] == "оплачен":
        month = row["order_date"][:7]
        month_revenue[month] = month_revenue.get(month, 0) + to_num(row["amount"])

for month in sorted(month_revenue):
    print(month, f"{month_revenue[month]:,.0f} тг")

september = month_revenue["2026-09"]
october = month_revenue["2026-10"]
change = (october - september) / september * 100
print(f"Изменение октября к сентябрю: {change:.1f}%")

2026-09 57,489,850 тг
2026-10 44,419,510 тг
Изменение октября к сентябрю: -22.7%


In [9]:
# 6

city_total = {}
city_cancelled = {}

for row in rows:
    city = clean_city(row["city"])
    city_total[city] = city_total.get(city, 0) + 1
    if row["status"] == "отменён":
        city_cancelled[city] = city_cancelled.get(city, 0) + 1

for city in sorted(city_total):
    cancelled = city_cancelled.get(city, 0)
    share = cancelled / city_total[city] * 100
    print(f"{city:<10} отменено {cancelled:>3} из {city_total[city]:<5} = {share:.1f}%")

Актобе     отменено  41 из 397   = 10.3%
Алматы     отменено 172 из 2188  = 7.9%
Астана     отменено 123 из 1390  = 8.8%
Атырау     отменено  11 из 146   = 7.5%
Караганда  отменено  48 из 538   = 8.9%
Шымкент    отменено 164 из 804   = 20.4%


In [10]:
# 7

customer_revenue = {}
for row in rows:
    if row["status"] == "оплачен":
        customer = row["customer_id"]
        customer_revenue[customer] = customer_revenue.get(customer, 0) + to_num(row["amount"])

sorted_customers = sorted(customer_revenue.items(), key=second_item, reverse=True)
top5 = sorted_customers[:5]

for customer, money in top5:
    print(f"{customer:<8}{money:>12,.0f} тг")

TEST      29,999,970 тг
C01627       257,180 тг
C00485       215,920 тг
C01338       192,080 тг
C02122       191,670 тг


**Ответ к задаче 7:** на первом месте клиент `TEST`. Его выручка очень сильно отличается от остальных клиентов, поэтому это похоже на тестовую или ошибочную запись, а не на обычного покупателя. В задачах 10 и 12 исключаем этого клиента из расчёта.

In [11]:
# 8

city_customers = {}
for row in rows:
    city = clean_city(row["city"])
    if city not in city_customers:
        city_customers[city] = set()
    city_customers[city].add(row["customer_id"])

for city in sorted(city_customers):
    print(city, len(city_customers[city]))

Актобе 358
Алматы 1351
Астана 981
Атырау 138
Караганда 451
Шымкент 655


## [Челлендж]

9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?

10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?

11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?

12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [12]:
# Челлендж

# 9

seen = set()
unique_rows = []
duplicate_rows = []

for row in rows:
    key = tuple(row.values())
    if key in seen:
        duplicate_rows.append(row)
    else:
        seen.add(key)
        unique_rows.append(row)

print("Всего строк:", len(rows))
print("Без дублей: ", len(unique_rows))
print("Дублей:     ", len(duplicate_rows))

duplicates_by_month = {}
for row in duplicate_rows:
    month = row["order_date"][:7]
    duplicates_by_month[month] = duplicates_by_month.get(month, 0) + 1

for month in sorted(duplicates_by_month):
    print(month, duplicates_by_month[month])

Всего строк: 5463
Без дублей:  5283
Дублей:      180
2026-09 180


**Ответ к задаче 9:** полных дублей **180**, и все они в **сентябре** (в октябре дублей нет).

In [13]:
# 10

clean_revenue = {}
for row in unique_rows:
    if row["status"] == "оплачен" and row["customer_id"] != "TEST":
        month = row["order_date"][:7]
        clean_revenue[month] = clean_revenue.get(month, 0) + to_num(row["amount"])

for month in sorted(clean_revenue):
    print(month, f"{clean_revenue[month]:,.0f} тг")

clean_september = clean_revenue["2026-09"]
clean_october = clean_revenue["2026-10"]
clean_change = (clean_october - clean_september) / clean_september * 100

print(f"Было (задача 5):    {change:.1f}%")
print(f"Стало после чистки: {clean_change:.1f}%")

2026-09 35,243,410 тг
2026-10 34,419,520 тг
Было (задача 5):    -22.7%
Стало после чистки: -2.3%


In [14]:
# 11

customer_orders = {}
for row in unique_rows:
    customer = row["customer_id"]
    customer_orders[customer] = customer_orders.get(customer, 0) + 1

best_customer = ""
best_count = 0
for customer in customer_orders:
    if customer_orders[customer] > best_count:
        best_count = customer_orders[customer]
        best_customer = customer

print(f"Больше всего заказов у клиента {best_customer}: {best_count}")

Больше всего заказов у клиента C01818: 10


**Задача 12 — сообщение менеджеру:**

Первый расчёт показывал, что выручка оплаченных заказов в октябре упала на 22,7 % по сравнению с сентябрём. На самом деле она снизилась всего на 2,3 % (с 35,2 до 34,4 млн тг). Расчёт был неверным по двум причинам: в сентябре в данных 180 полностью повторяющихся строк (дубли заказов), а ещё в выручку попали три тестовых заказа клиента `TEST` по 9,99 млн тг каждый, из них два в сентябре и один в октябре. Из-за этого сентябрь получился сильно завышенным, и падение выглядело намного больше, чем есть на самом деле.